# 🤖 Agenten: Emergentes Tool-Verhalten

Der Agent wechselt zwischen Nachdenken und Tool-Aufrufen. Aber die eigentliche Magie: **Die Optimierung verbessert nicht nur die Worte, sondern auch WIE der Agent seine Werkzeuge einsetzt** — die Entscheidungsstrategie!

## Was ist ein Agent?

Bisher haben unsere Modelle nur Text rein → Text raus gemacht. Ein **Agent** kann mehr: Er bekommt **Tools** (Taschenrechner, Datenbank-Suche, etc.) und entscheidet **selbst**, welches Tool er wann einsetzt.

Das Spannende: Dieses Verhalten ist **emergent** — wir programmieren nicht "wenn Mathe-Frage, dann Taschenrechner". Der Agent lernt das selbst.

In diesem Notebook:
1. Wir probieren die Tools einzeln aus
2. Wir lassen den Agenten selbst entscheiden
3. Wir optimieren sein Verhalten — ja, auch das geht!


In [ ]:
import sys; sys.path.insert(0, ".")
from dspy_tasks.config import configure_dspy
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.tools import TOOL_REGISTRY, calculate, search_tickets, get_ticket_stats
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODEL = "github_copilot/gpt-4o-mini"
configure_dspy(MODEL)


### 💡 Warum GPT-4o-mini statt GPT-5.1?

Wir verwenden hier absichtlich ein **schwächeres und schnelleres Modell**. GPT-5.1 löst diese Aufgaben fast perfekt — aber es ist auch **langsamer und teurer**. In der Praxis stellt sich oft die Frage:

> *Brauche ich das teuerste Modell? Oder reicht ein günstigeres Modell + gutes Prompt-Engineering?*

Genau das zeigen wir hier: **Optimierung kann ein günstigeres Modell auf das Niveau eines teureren bringen** — bei niedrigeren Kosten und schnellerer Ausführung. Das ist einer der wichtigsten ROI-Hebel in der Produktion.

## 🔧 Die verfügbaren Tools

Unsere Agenten haben Zugriff auf echte Tool-Funktionen. Lass uns sie erstmal direkt ausprobieren, bevor wir sie den Agenten geben.

### 🔄 Der Agent-Loop

So arbeitet ein Agent: Er **denkt nach** (was ist die Frage?), **wählt ein Tool** (brauche ich den Taschenrechner oder die Suche?), **führt es aus**, und **prüft das Ergebnis** (bin ich fertig oder muss ich nochmal nachdenken?).

Dieser Loop kann mehrere Runden dauern — bei komplexen Fragen kombiniert der Agent mehrere Tools nacheinander.


In [ ]:
diagram([
    {"label": "Denken", "detail": "Agent überlegt", "icon": "🤔", "color": "#0078d4"},
    {"label": "Tool wählen", "detail": "search? calculate?", "icon": "🔧", "color": "#ca5010"},
    {"label": "Ausführen", "detail": "Tool aufrufen", "icon": "⚡", "color": "#ca5010"},
    {"label": "Ergebnis prüfen", "detail": "Nochmal? Fertig?", "icon": "🔍", "color": "#107c10"},
], title="Der ReAct-Loop: Denken → Handeln → Beobachten")

### 🔧 Erst die Tools kennenlernen

Bevor wir sie dem Agenten geben, probieren wir die Tools selbst aus. So siehst du, was der Agent zur Verfügung hat:

- **calculate** — kann beliebige Mathe-Ausdrücke berechnen
- **search_tickets** — durchsucht die echten Ticket-Daten nach Stichworten
- **get_ticket_stats** — liefert Statistiken (z.B. Verteilung nach Priorität)

Klick Run und sieh dir die Ergebnisse an. Genau diese Ergebnisse bekommt nachher auch der Agent.


## 🧪 Tools direkt ausprobieren

Jedes Tool ist eine einfache Python-Funktion. Der Agent entscheidet selbst, welches er wann aufruft — aber lass uns sie erstmal einzeln testen.


In [ ]:
# Tools are just Python functions — try them!
calc_result = calculate("(45 * 3) + 17")
search_result = search_tickets("network")
stats_result = get_ticket_stats("priority")

print("🧮 calculate('(45 * 3) + 17'):")
print(f"   → {calc_result}")
print()
print("🔍 search_tickets('network'):")
print(f"   → {search_result}")
print()
print("📊 get_ticket_stats('priority'):")
print(f"   → {stats_result}")

## Task 16: Calculator Agent — Der einfachste Agent

Der Calculator Agent bekommt eine Mathe-Frage und hat ein `calculate`-Tool zur Verfügung. Er muss selber entscheiden, wann und wie er es benutzt.

In [ ]:
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy
MODELS = get_available_models()


print(f"⏳ Running Calculator Agent on {MODEL}...")
result = run_baseline("calculator_agent", max_eval=5)
display_score("Calculator Agent", result.score)
display_results_table(result.individual_scores)


### 🔍 Warum ist der Search Agent schwieriger?

Der Calculator Agent hat ein einfaches Tool: Rechnung rein, Ergebnis raus. Der Search Agent muss:

1. Die richtige **Suchanfrage formulieren** (nicht trivial!)
2. Die **Ergebnisse interpretieren** (was ist relevant?)
3. Eventuell **nochmal suchen** mit anderen Stichworten
4. Alles zu einer **kohärenten Antwort** zusammenfassen

Das ist wie der Unterschied zwischen "Was ist 2+2?" und "Finde heraus, welches Team die meisten Netzwerk-Tickets hat."


## Task 17: Search & Synthesize — Deine Daten durchsuchen

Jetzt wird's interessant: der Agent durchsucht die echte Ticket-Datenbank und fasst Antworten zusammen. Er entscheidet selbst, welche Tools er braucht.

In [ ]:
print(f"⏳ Search Agent läuft...")
print("   Der Agent durchsucht die Ticket-Datenbank...\n")
search_baseline = run_baseline("search_agent", max_eval=10)
display_score("Search Agent", search_baseline.score)
display_results_table(search_baseline.individual_scores)

### 🔴 Warum ist der Score so niedrig?

Schau dir die Ergebnisse genau an — der Agent findet oft die **richtigen Zahlen**, aber er verpackt sie in ganze Sätze:

| Erwartet | Agent gibt zurück | Score |
|---|---|---|
| `99` | `The total number of tickets is 99 (60 Passwort tickets + 39 VPN tickets).` | 13% |
| `7.5` | `There are 224 tickets with the status 'Assigned', which is 7.5% of all 3000 tickets.` | 0% |
| `23` | `There are a combined total of 23 tickets for Netzwerk and WLAN (19 + 4).` | 12% |

Die **Metrik** berechnet einen Token-F1-Score zwischen der erwarteten und der tatsächlichen Antwort. Wenn die erwartete Antwort `"99"` ist und der Agent `"The total number of tickets is 99 (60 Passwort + 39 VPN tickets)."` zurückgibt, ist `"99"` nur ein winziger Teil der Antwort — das ergibt ~13%.

Zwei Probleme gleichzeitig:
1. **Format:** Der Agent antwortet in ganzen Sätzen statt nur mit der Zahl
2. **Sources:** Die Metrik erwartet `"ticket_database"`, aber der Agent schreibt z.B. `"Search results for Netzwerk and WLAN tickets"`

**Das ist genau das, was Optimierung fixen kann:**
- Ein **besserer Prompt** sagt dem Modell "antworte nur mit der Zahl"
- **Few-Shot Beispiele** zeigen dem Modell das erwartete kurze Format
- **MIPROv2** kann die Instruktionen automatisch umschreiben

## 🎯 Verbesserung Schritt 1: Besserer Prompt

Die Signature ist absichtlich vage ("Synthesized answer from search results"). Das Modell weiss nicht, dass es kurz antworten soll. 

Was passiert, wenn wir dem Agenten einfach einen besseren Prompt geben — z.B. "antworte nur mit der Zahl"?

In [ ]:
from dspy_tasks.actions import run_with_prompt

BETTER_PROMPT = "Search the ticket database to answer this question. Answer with ONLY the number or short fact — no full sentences. For sources, always write: ticket_database"

print("⏳ Search Agent mit besserem Prompt...")
improved_result = run_with_prompt("search_agent", BETTER_PROMPT, max_eval=10)
display_score("Search Agent (besserer Prompt)", improved_result.score)
display_results_table(improved_result.individual_scores)

print(f"\n📊 Vergleich:")
print(f"   Vorher (vager Prompt):      {search_baseline.score:.0%}")
print(f"   Nachher (präziser Prompt):   {improved_result.score:.0%}")
display_improvement(search_baseline.score, improved_result.score)

## ⚙️ Verbesserung Schritt 2: MIPROv2 Optimierung

Manchmal ist es gar nicht so einfach, den richtigen Prompt von Hand zu schreiben. Bei komplexen Aufgaben weisst du vielleicht nicht genau, *welche* Formulierung den grössten Unterschied macht — oder der Prompt wird so lang und spezifisch, dass er bei leicht anderen Fragen wieder bricht.

Genau hier hilft **MIPROv2**: Er probiert systematisch verschiedene Instruktionen durch und fügt **Few-Shot Beispiele** hinzu, die dem Agenten das erwartete Antwortformat zeigen. Du gibst ihm nur den besten Prompt, den du hast, und er sucht automatisch nach einer noch besseren Variante.

Schauen wir, ob MIPROv2 mehr rausholen kann als unser manueller Versuch:

In [ ]:
BAD_PROMPT = "Search the ticket database to answer this question."

print("⏳ MIPROv2 optimiert den Search Agent...")
print("   Basis: der vage Default-Prompt (NICHT unser verbesserter!)")
print("   Das kann 1-2 Minuten dauern...\n")

mipro_result = run_optimization("search_agent", "MIPROv2", max_eval=10, instructions=BAD_PROMPT)

print("━" * 60)
display_score("Search Agent: Nach MIPROv2", mipro_result.optimized_score)
if mipro_result.optimized_individual_scores:
    display_results_table(mipro_result.optimized_individual_scores)

# --- Alle 3 Schritte vergleichen ---
print(f"\n{'━'*60}")
print("📊 Alle 3 Ansätze im Vergleich:")
print(f"   1. Zero-Shot (vager Prompt):        {search_baseline.score:.0%}")
print(f"   2. Manuell besserer Prompt:          {improved_result.score:.0%}")
print(f"   3. MIPROv2 (vom vagen Prompt!):      {mipro_result.optimized_score:.0%}")

display_improvement(search_baseline.score, mipro_result.optimized_score)

print("\n📝 MIPROv2 — Was hat er aus dem vagen Prompt gemacht?")
print(mipro_result.prompt_after[:2000])
if len(mipro_result.prompt_after) > 2000:
    print(f"... ({len(mipro_result.prompt_after)} Zeichen)")

display_prompt_diff(BAD_PROMPT, mipro_result.prompt_after)

display_insight("Was gerade passiert ist",
    f"Zero-Shot: {search_baseline.score:.0%} → Manuell: {improved_result.score:.0%} → "
    f"MIPROv2 (vom vagen Prompt): {mipro_result.optimized_score:.0%}. "
    + (f"Der manuelle Prompt hat MIPROv2 geschlagen! Das zeigt: wenn du genau weisst, "
       "was das Problem ist (hier: Antwortformat), ist ein gezielter manueller Fix oft besser "
       "als ein automatischer Optimizer, der vom schlechtesten Punkt startet. "
       "MIPROv2 hat sich zwar verbessert ({search_baseline.score:.0%} → {mipro_result.optimized_score:.0%}), "
       "aber die riesigen Few-Shot Beispiele im Prompt machen ihn langsamer und teurer."
       if improved_result.score > mipro_result.optimized_score else
       f"MIPROv2 hat automatisch herausgefunden, dass kurze Antworten besser scoren — "
       "ohne dass wir ihm das gesagt haben!"))

### 📝 Was du mitnehmen solltest

1. **Agenten = LLM + Tools** — Das Modell entscheidet selbst, welches Tool es nutzt
2. **Emergentes Verhalten** — Wir programmieren keine Regeln, der Agent lernt sie
3. **Format matters!** — Ein Agent kann die richtige Antwort finden aber trotzdem 0% scoren, weil das Ausgabeformat nicht stimmt
4. **OutputField-Beschreibungen sind dein Steuerrad** — Präzise `desc`-Felder in der Signature können den Score dramatisch verbessern
5. **MIPROv2 optimiert auch Agenten** — Nicht nur was der Agent sagt, sondern wie er vorgeht und wie er seine Antworten formatiert

> **Die Lektion:** Wenn dein Agent die richtigen Infos findet aber schlecht scored, schau zuerst auf die **Metrik** und das **Ausgabeformat**. Oft ist das Problem nicht das Wissen, sondern die Verpackung.

## ⏭️ Das Finale!

Agenten können Tools nutzen UND ihr Verhalten kann optimiert werden. Du hast jetzt alle Bausteine gesehen:

1. ✅ Metriken und Evaluation
2. ✅ Automatische Prompt-Optimierung
3. ✅ Domain-Daten als Wettbewerbsvorteil
4. ✅ Agenten-Optimierung

**Zeit für das Gesamtbild und das Quiz!** → Notebook 05

👉 **[Weiter zu Notebook: Das Gesamtbild →](05_full_picture.ipynb)**